# Single Pendulum Test

Gonna do a more neat based algorithm or evolutionary based since I don't really have a good cost equation and more of a test pendulum type? type?

In [1]:
import numpy as np
import plotly.graph_objects as go
from ipywidgets import FloatSlider, VBox
import threading
import time

## Physics Simulation

Derivation for the acceleration is
$$
I\alpha = \tau_{net} \\
I\alpha = \tau_{gravity} + \tau_{inertial} \\
I\alpha = mgL \sin\theta + ma_{c}L \cos\theta \\
mL^2 \alpha = mgL \sin\theta + ma_{c}L \cos\theta \\
\alpha = \frac{mgL \sin\theta + ma_{c}L \cos\theta}{mL^2} \\
$$
$$\alpha = \frac{g \sin\theta + a_{c} \cos\theta}{L}$$

Then we need to numerically integrate, gonna use semi euler not verlet which is what I was doing but would rather have the direct v and $\omega$ than implicit ones
Last time I used loops since I didn't know linear algebra this but tbh there isn't going to many pendulums like balls so not as impactful
Also theta her is going to be based of physics and math way or of the y axis but gonna make 0 the top or where I want the output to converge too since I think that's easier than converging to a arbitrary angle float of $\pi$

In [2]:
class Cart_Pole_Sim:
    def __init__(
        self,
        dt: float = 0.02,
        x: float = 0.0,
        v: float = 0.0,
        theta: float = np.pi / 2,
        omega: float = 0.0,
        l: float = 1.0,
        m: float = 1.0,
        g: float = 9.81,
    ):
        self.dt = dt

        self.g = g
        self.l = l
        self.m = m

        self.x = x
        self.v = v

        self.theta = theta
        self.omega = omega

    def step(self, cart_accel: float):
        alpha = (self.g * np.sin(self.theta) - cart_accel * np.cos(self.theta)) / self.l

        self.omega += alpha * self.dt
        self.theta += self.omega * self.dt
        self.theta = ((self.theta + np.pi) % (2 * np.pi)) - np.pi # Wrap between Pi and -Pi

        self.v += cart_accel * self.dt
        self.x += self.v * self.dt

In [3]:
t = 0.0
dt = 0.1
time_running = 1
running = True

simple_cart = Cart_Pole_Sim(dt)

current_time = time.perf_counter()
accumulated_time = 0.0

while running:
    new_time = time.perf_counter()
    step_time = new_time - current_time
    current_time = new_time

    accumulated_time += step_time

    while accumulated_time >= dt:
        cart_accel = 0.0
        simple_cart.step(cart_accel)

        accumulated_time -= dt
        t += dt

    if t >= time_running:
        print(t)
        running = False

print(simple_cart.x, simple_cart.v, simple_cart.theta, simple_cart.omega)

1.0999999999999999
0.0 0.0 -1.5641871789483894 0.8235091176906693


### Lets Render Every nth Frame

In [4]:
t = 0.0
dt = 0.05
time_running = 3
running = True

simple_cart = Cart_Pole_Sim(dt, 0.0, 0.0, .5)

current_time = time.perf_counter()
accumulated_time = 0.0
steps = 0

history: list[dict[str, float]] = []

while running:
    new_time = time.perf_counter()
    step_time = new_time - current_time
    current_time = new_time

    accumulated_time += step_time

    while accumulated_time >= dt:
        cart_accel = 0.0
        simple_cart.step(cart_accel)

        accumulated_time -= dt
        t += dt

        steps += 1
        history.append({"frame": steps, "x": simple_cart.x, "theta": simple_cart.theta})

    if t >= time_running:
        print(t)
        running = False

print(simple_cart.x, simple_cart.v, simple_cart.theta, simple_cart.omega)

3.049999999999997
0.0 0.0 1.2462027276407204 -3.5494427968447386


In [5]:
def plot_simulation_ghosting(
    history: list[dict[str, float]],
    pole_length: float = 1.0,
    title: str = "Inverted Pendulum Ghosting",
):
    fig = go.Figure()

    # Draw the ground line
    fig.add_shape(
        type="line",
        x0=-5,
        y0=0,
        x1=5,
        y1=0,
        line=dict(color="Gray", width=1, dash="dash"),
    )

    for i, entry in enumerate(history):
        total_ghosts = len(history)

        cart_x = entry["x"]
        theta = entry["theta"]

        # 0 radians is up so x = sin, y = cos
        pole_x = cart_x + pole_length * np.sin(theta)
        pole_y = pole_length * np.cos(theta)

        opacity = max(0.1, i / total_ghosts)

        fig.add_trace(
            go.Scatter(
                x=[cart_x, pole_x],
                y=[0, pole_y],
                mode="lines+markers",
                line=dict(width=2, color="blue"),
                marker=dict(size=4, color="red"),
                opacity=opacity,
                showlegend=False,
                hoverinfo="text",
                text=f"Frame: {entry['frame']} | Theta: {np.degrees(theta):.1f}°",
            )
        )

        fig.add_trace(
            go.Scatter(
                x=[cart_x],
                y=[0],
                mode="markers",
                marker=dict(symbol="square", size=10, color="black"),
                opacity=opacity,
                showlegend=False,
            )
        )

    fig.update_layout(
        title=title,
        xaxis=dict(title="Cart Position (x)", range=[-2.5, 2.5]),
        yaxis=dict(
            title="Height (y)", range=[-1.5, 1.5], scaleanchor="x", scaleratio=1
        ),
        template="plotly_white",
        width=800,
        height=500,
    )

    return fig

plot_simulation_ghosting(history[::5], simple_cart.l)

In [6]:
def animate_simulation(
    history: list[dict[str, float]],
    pole_length: float = 1.0,
    title: str = "Inverted Pendulum Animation",
):
    start = history[0]
    fig = go.Figure(
        data=[
            go.Scatter(
                x=[start["x"]],
                y=[0],
                mode="markers",
                marker=dict(symbol="square", size=20, color="black"),
                name="Cart",
            ),
            go.Scatter(
                x=[start["x"], start["x"] + pole_length * np.sin(start["theta"])],
                y=[0, pole_length * np.cos(start["theta"])],
                mode="lines+markers",
                line=dict(width=4, color="blue"),
                marker=dict(size=8, color="red"),
                name="Pole",
            ),
        ],
        layout=go.Layout(
            xaxis=dict(range=[-2.5, 2.5], autorange=False),
            yaxis=dict(
                range=[-1.5, 1.5], autorange=False, scaleanchor="x", scaleratio=1
            ),
            title=title,
            template="plotly_white",
            width=800,
            height=500,
            updatemenus=[
                dict(
                    type="buttons",
                    buttons=[
                        dict(
                            label="Play",
                            method="animate",
                            args=[
                                None,
                                {
                                    "frame": {"duration": 20, "redraw": False},
                                    "fromcurrent": True,
                                },
                            ],
                        ),
                        dict(
                            label="Pause",
                            method="animate",
                            args=[
                                [None],
                                {
                                    "frame": {"duration": 0, "redraw": False},
                                    "mode": "immediate",
                                    "fromcurrent": True,
                                },
                            ],
                        ),
                    ],
                )
            ],
        ),
        frames=[
            go.Frame(
                data=[
                    go.Scatter(x=[e["x"]], y=[0]),
                    go.Scatter(
                        x=[e["x"], e["x"] + pole_length * np.sin(e["theta"])],
                        y=[0, pole_length * np.cos(e["theta"])],
                    ),
                ],
                name=str(i),
            )
            for i, e in enumerate(history)
        ],
    )

    return fig


animate_simulation(history, simple_cart.l)

## Real Time

In [7]:
if "physics_thread" in globals() and physics_thread is not None:
    physics_thread.is_alive()
    physics_thread.join(timeout=0.2)

physics_stop_event = threading.Event()

dt = 0.05
simple_cart = Cart_Pole_Sim(dt)
accumulated_time = 0.0
current_time = time.perf_counter()

fig = go.FigureWidget()
fig = go.FigureWidget()
fig.add_scatter(
    x=[0], y=[0], mode="markers", marker=dict(symbol="square", size=30, color="black")
)
fig.add_scatter(x=[0, 0], y=[0, 0], mode="lines", line=dict(width=4, color="blue"))
fig.add_scatter(x=[0], y=[0], mode="markers", marker=dict(size=12, color="red"))
fig.update_layout(
    xaxis=dict(range=[-2.5, 2.5], autorange=False),
    yaxis=dict(range=[-1.5, 1.5], autorange=False, scaleanchor="x", scaleratio=1),
    template="plotly_white",
    width=700,
    height=400,
    showlegend=False,
)

slider = FloatSlider(value=0, min=-4, max=4, step=0.01, description="Move Cart")


def physics_loop(stop_event):
    global current_time, accumulated_time  # Assignment not mutation like other cart
    print("New thread started.")

    while not stop_event.is_set():
        new_time = time.perf_counter()
        step_time = new_time - current_time
        current_time = new_time

        accumulated_time += step_time

        c_x = slider.value
        theta = simple_cart.theta

        while accumulated_time >= dt:
            v_now = (c_x - simple_cart.x) / dt * .1
            a_now = (v_now - simple_cart.v) / dt

            simple_cart.step(a_now)
            accumulated_time -= dt

            simple_cart.x = c_x
            simple_cart.v = v_now

            simple_cart.omega = simple_cart.omega * .999

        b_x = c_x + simple_cart.l * np.sin(theta)
        b_y = simple_cart.l * np.cos(theta)

        # 4. Batch Update (Check if data exists to prevent IndexError)
        if len(fig.data) >= 3:
            with fig.batch_update():
                fig.data[0].x = [c_x]
                fig.data[1].x, fig.data[1].y = [c_x, b_x], [0, b_y]
                fig.data[2].x, fig.data[2].y = [b_x], [b_y]

        time.sleep(0.04)


physics_thread = threading.Thread(
    target=physics_loop, args=(physics_stop_event,), daemon=True
)
physics_thread.start()

VBox([fig, slider])

New thread started.


    'data': [{'marker': {'color': 'black', 'size': 30, 'symbol': 'square'},
    …